# QLoRA fine-tune — Qwen2.5-3B-Instruct incident reportsThin runner. **All training logic lives in `src/train_qlora.py`** — this notebookinstalls dependencies, locates the data, and calls it. Do not paste training codein here; two copies will drift.## Before running1. Session settings → **Accelerator: GPU T4 x2 or P100**, **Internet: On**   (both require a phone-verified Kaggle account).2. Add the project as a Kaggle Dataset input containing `train.jsonl`,   `test.jsonl`, and the `src/` directory.3. Set `DATASET_SLUG` below to match the input path shown in the sidebar.Expect roughly 40–70 minutes for 3 epochs over 360 examples at 1536 tokens on asingle T4. That is within the 12-hour kernel ceiling, but the run is batch-onlywith no interactivity — if it dies, it dies. Adapters are saved every epoch so acrash at epoch 3 still leaves epoch 2 usable.

In [ ]:
# 1. Dependencies. Kaggle images ship torch + CUDA; these are the additions.!pip install -q -U "transformers>=4.44" "peft>=0.12" "trl>=0.9" \    "bitsandbytes>=0.43" "accelerate>=0.33" "datasets>=2.20" sentencepiece

In [ ]:
# 2. Locate the input dataset and the project source.import os, shutil, subprocess, sysfrom pathlib import PathDATASET_SLUG = "telemetry-incident-report"   # <-- match your Kaggle input nameINPUT = Path("/kaggle/input") / DATASET_SLUGWORK = Path("/kaggle/working")assert INPUT.exists(), f"{INPUT} not found. Check the input name in the sidebar: " \                       f"{sorted(p.name for p in Path('/kaggle/input').iterdir())}"# Copy src/ into the working dir so `python -m src.train_qlora` resolves, and so# the read-only input mount is never written to.if (INPUT / "src").exists():    shutil.copytree(INPUT / "src", WORK / "src", dirs_exist_ok=True)DATA = WORK / "data" / "processed"DATA.mkdir(parents=True, exist_ok=True)for name in ("train.jsonl", "test.jsonl"):    src = INPUT / name    if not src.exists():        src = INPUT / "data" / "processed" / name    shutil.copy(src, DATA / name)os.chdir(WORK)print("cwd:", Path.cwd())print("train:", sum(1 for _ in open(DATA / "train.jsonl")), "examples")print("test :", sum(1 for _ in open(DATA / "test.jsonl")), "examples")

In [ ]:
# 3. Confirm the GPU before spending an hour on it.import torchprint("cuda:", torch.cuda.is_available())if torch.cuda.is_available():    print("device:", torch.cuda.get_device_name(0))    print("memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")    print("bf16 supported:", torch.cuda.is_bf16_supported())else:    raise RuntimeError("No GPU. Set Accelerator to GPU in session settings.")

In [ ]:
# 4. Smoke test first — 8 examples, 2 steps. Proves the pipeline and the loss#    masking before committing to the full run.!python -m src.train_qlora --smoke --train data/processed/train.jsonl

In [ ]:
# 5. The real run. Hyperparameters are constants at the top of src/train_qlora.py.!python -m src.train_qlora \    --train data/processed/train.jsonl \    --out /kaggle/working/qlora-adapter

In [ ]:
# 6. Check what was written. The adapter is a few tens of MB, not a full model.import jsonfrom pathlib import PathOUT = Path("/kaggle/working/qlora-adapter")for p in sorted(OUT.rglob("*")):    if p.is_file():        print(f"{p.relative_to(OUT)}  {p.stat().st_size / 1e6:.1f} MB")summary = json.loads((OUT / "training_summary.json").read_text())print()print("steps          :", summary["steps"])print("elapsed        :", round(summary["elapsed_seconds"] / 60, 1), "min")print("final train loss:", summary["final_train_loss"])print("final eval loss :", summary["final_eval_loss"])

In [ ]:
# 7. Predictions for both variants, so the comparison in SCOPE.md 5.3 uses one#    harness. Baseline first — it needs no adapter.!python -m src.generate_predictions --model base  --load-4bit \    --test data/processed/test.jsonl!python -m src.generate_predictions --model tuned --load-4bit \    --adapter /kaggle/working/qlora-adapter \    --test data/processed/test.jsonl

## OutputsEverything under `/kaggle/working` is downloadable from the kernel's Output tab:- `qlora-adapter/` — LoRA weights, tokenizer, `training_summary.json`- `data/processed/preds_base.jsonl`, `preds_tuned.jsonl`Scoring runs locally against those prediction files — it needs no GPU.